# Bài 7 — Giám sát theo vùng: PolygonZone

**Mục tiêu:** Đếm/giám sát đối tượng **bên trong vùng đa giác** (làn đường, bãi đỗ, vùng cấm) — số xe trong vùng hiện ngay giữa polygon trên cửa sổ.

## 0. Chuẩn bị (asset video + display.py dùng chung)

In [1]:
!pip install -q supervision ultralytics "supervision[assets]"

In [2]:
from supervision.assets import download_assets, VideoAssets

download_assets(VideoAssets.VEHICLES)
print(VideoAssets.VEHICLES.value)  # "vehicles.mp4"

[2026-08-18 17:18:25] [INFO] supervision.assets.downloader - vehicles.mp4 asset download complete.
vehicles.mp4


In [3]:
%%writefile display.py
# display.py — hàm hiển thị dùng chung cho toàn giáo trình
import cv2

WINDOW_NAME = "Supervision - Live"
MAX_DISPLAY_WIDTH = 1280   # thu nhỏ frame cho vừa màn hình (chỉ để XEM, không ảnh hưởng xử lý)


def show_frame(frame, window_name: str = WINDOW_NAME, wait: int = 1) -> bool:
    """Hiện frame lên cửa sổ. Trả về False nếu người dùng bấm Q/ESC (muốn thoát).

    wait=1  -> dùng cho video (hiện liên tục, không chặn)
    wait=0  -> dùng cho ảnh tĩnh (dừng lại chờ bấm phím bất kỳ)
    """
    h, w = frame.shape[:2]
    if w > MAX_DISPLAY_WIDTH:                      # thu nhỏ để vừa màn hình
        scale = MAX_DISPLAY_WIDTH / w
        frame = cv2.resize(frame, (int(w * scale), int(h * scale)))

    cv2.imshow(window_name, frame)
    key = cv2.waitKey(wait) & 0xFF
    if key in (ord("q"), ord("Q"), 27):            # Q hoặc ESC -> thoát
        return False
    return True


def close_windows():
    cv2.destroyAllWindows()

Overwriting display.py


## 7.1. Pipeline PolygonZone — giữ khung Bài 4/5

>  Muốn lấy tọa độ polygon chuẩn theo video của bạn: dùng lại tool click chuột ở Bài 6.2, hoặc [polygonzone.roboflow.com](https://polygonzone.roboflow.com).

In [4]:
import numpy as np
import cv2
import supervision as sv
from ultralytics import YOLO
from display import show_frame, close_windows

SOURCE_VIDEO = "vehicles.mp4"
TARGET_VIDEO = "bai7_output.mp4"
VEHICLE_CLASSES = [2, 3, 5, 7]
MAX_FRAMES = 300   # test nhanh trước; đặt None để chạy hết video

model = YOLO("yolov8n.pt")
CLASS_NAMES = model.names   # dict {class_id: ten_class} — dùng thay vì detections.data["class_name"]
# (data["class_name"] bị ByteTrack loại bỏ sau khi update_with_detections, nên không dùng được nữa)
video_info = sv.VideoInfo.from_video_path(SOURCE_VIDEO)
W, H = video_info.width, video_info.height

#  Chỉnh 2 polygon này theo video thực tế (dùng tool click chuột ở Bài 6.2)
polygon_lane_1 = np.array([[0, H//2], [W//2, H//2], [W//2, H], [0, H]])
polygon_lane_2 = np.array([[W//2, H//2], [W, H//2], [W, H], [W//2, H]])

zones = [
    sv.PolygonZone(polygon=polygon_lane_1, triggering_anchors=(sv.Position.BOTTOM_CENTER,)),
    sv.PolygonZone(polygon=polygon_lane_2, triggering_anchors=(sv.Position.BOTTOM_CENTER,)),
]
zone_annotators = [
    sv.PolygonZoneAnnotator(zone=zones[0], color=sv.Color.RED, thickness=2, text_scale=1, opacity=0.2),
    sv.PolygonZoneAnnotator(zone=zones[1], color=sv.Color.BLUE, thickness=2, text_scale=1, opacity=0.2),
]

tracker = sv.ByteTrack(frame_rate=video_info.fps)
box_annotator = sv.BoxAnnotator(thickness=2, color_lookup=sv.ColorLookup.TRACK)
label_annotator = sv.LabelAnnotator(text_scale=0.5, color_lookup=sv.ColorLookup.TRACK)

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_18636\2382094896.py:31: FutureWarning: The `ByteTrack` was deprecated since v0.28.0. It will be removed in v0.31.0.
  tracker = sv.ByteTrack(frame_rate=video_info.fps)


## 7.2. Cảnh báo dừng lâu (dwell time) — banner đỏ trên cửa sổ

In [5]:
from collections import defaultdict

frames_in_zone = defaultdict(int)
DWELL_SECONDS = 10   # ngưỡng cảnh báo dừng quá lâu


def check_dwell(frame, detections, zone, fps):
    in_zone = zone.trigger(detections)
    warnings = []
    for tid in detections.tracker_id[in_zone]:
        frames_in_zone[tid] += 1
        if frames_in_zone[tid] > fps * DWELL_SECONDS:
            warnings.append(int(tid))
    if warnings:
        cv2.rectangle(frame, (0, 0), (frame.shape[1], 90), (0, 0, 255), -1)
        cv2.putText(frame, f"!!! XE DUNG QUA LAU: {warnings} !!!",
                    (30, 60), cv2.FONT_HERSHEY_SIMPLEX, 1.5, (255, 255, 255), 3)
    return frame

## 7.3. Hàm xử lý frame + vòng lặp chính

In [6]:
def process_frame(frame: np.ndarray) -> np.ndarray:
    results = model(frame, imgsz=640, verbose=False)[0]
    detections = sv.Detections.from_ultralytics(results)
    detections = detections[np.isin(detections.class_id, VEHICLE_CLASSES)]
    detections = tracker.update_with_detections(detections)

    labels = [f"#{tid} {CLASS_NAMES[cid]}" for tid, cid
              in zip(detections.tracker_id, detections.class_id)]

    annotated = frame.copy()
    annotated = box_annotator.annotate(annotated, detections)
    annotated = label_annotator.annotate(annotated, detections, labels=labels)

    # Vẽ từng zone + tự động hiện current_count giữa vùng
    for zone, zone_annotator in zip(zones, zone_annotators):
        zone.trigger(detections)
        annotated = zone_annotator.annotate(annotated)
        annotated = check_dwell(annotated, detections, zone, video_info.fps)

    return annotated

In [7]:
tracker.reset()

sink = sv.VideoSink(target_path=TARGET_VIDEO, video_info=video_info)
sink.__enter__()
try:
    for i, frame in enumerate(sv.get_video_frames_generator(SOURCE_VIDEO)):
        if MAX_FRAMES is not None and i >= MAX_FRAMES:
            break
        annotated = process_frame(frame)
        sink.write_frame(annotated)
        if not show_frame(annotated):
            print("Nguoi dung bam Q - dung som.")
            break
finally:
    sink.__exit__(None, None, None)
    close_windows()

print("Xong! Video da luu tai:", TARGET_VIDEO)

Xong! Video da luu tai: bai7_output.mp4


##  Checkpoint Bài 7

Cửa sổ live hiện 2 polygon (2 làn đường), mỗi vùng hiện số xe hiện tại ngay giữa vùng; khi có xe dừng > 10 giây, banner đỏ cảnh báo hiện trên đầu khung hình.